# UT3.1 — Calidad y fiabilidad de los datos (Big Data Aplicado)

> Objetivo: entender **calidad** vs **fiabilidad** y practicar cómo **medir** precisión, completitud y consistencia con ejemplos sencillos.

**Qué vas a aprender**
- Diferencia entre **calidad** y **fiabilidad** (no son lo mismo).
- 3 dimensiones mínimas de calidad: **precisión**, **completitud**, **consistencia**.
- Patrones típicos de degradación en entornos distribuidos: duplicados por reintentos, huecos temporales, errores de unidades/zona horaria, cambios de esquema, escrituras parciales.

📌 Idea clave: en distribuido, lo “normal” es que existan retrasos, reintentos y estados intermedios; por eso hay que **diseñar y medir** la calidad.


## 0) Preparación

Vamos a trabajar con un dataset de ejemplo (inventado) de una app de movilidad (bicis/patinetes):
- eventos de viaje (start/end),
- telemetría (GPS),
- pagos.

Lo haremos a propósito con errores típicos:
- valores imposibles (precisión),
- campos vacíos y huecos de tiempo (completitud),
- duplicados por reintentos (consistencia),
- euros vs céntimos y UTC vs hora local (errores silenciosos).


In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

pd.set_option("display.max_columns", None)
np.random.seed(7)


## 1) Crear un dataset con problemas reales

En un sistema distribuido los datos pasan por varios pasos (app → cola → procesamiento → lake → BI).
En ese camino pueden aparecer fallos **silenciosos** (no rompen el job, pero rompen el significado).


In [ ]:
# =========================
# Simulación de eventos de viaje con "fallos" intencionados de calidad de datos
# =========================

# Importante (se asume que ya has importado):
# import pandas as pd
# import numpy as np
# from datetime import datetime, timedelta, timezone

# Generamos eventos de viaje (simulados)
n = 18  # número de eventos a generar

# Timestamp base en UTC (buenas prácticas: almacenar siempre en UTC en el data lake/warehouse)
base = datetime(2026, 2, 1, 8, 0, 0, tzinfo=timezone.utc)

# Creamos un DataFrame con datos sintéticos (aleatorios) para emular una tabla de eventos de movilidad
events = pd.DataFrame({
    # Identificador único de evento (en teoría debería ser único; luego metemos un duplicado a propósito)
    "event_id": [f"evt_{i:03d}" for i in range(n)],

    # Usuario asociado al evento (elegido aleatoriamente entre 4 usuarios)
    "user_id": np.random.choice([101, 102, 103, 104], size=n),

    # Vehículo asociado (bicis/patinete), aleatorio
    "vehicle_id": np.random.choice(["bike_1", "bike_2", "scoot_1"], size=n),

    # Timestamps en UTC (lo correcto): cada evento ocurre 5 minutos después del anterior
    "ts_utc": [base + timedelta(minutes=5*i) for i in range(n)],

    # Velocidad simulada: normal con media 18 km/h y desviación 6 (redondeada a 1 decimal)
    "speed_kmh": np.random.normal(loc=18, scale=6, size=n).round(1),

    # Precio en CÉNTIMOS (dato común en sistemas de pagos); a veces se confunde con euros
    "price_cents": np.random.choice([150, 200, 250, 300, 450], size=n),

    # Tipo de evento: inicio o fin de viaje con probabilidad 55% / 45%
    "event_type": np.random.choice(["trip_start", "trip_end"], size=n, p=[0.55, 0.45]),
})

# -------------------------
# 1) Precisión: introducimos valores imposibles (simulan sensor/ETL roto)
# -------------------------

# Velocidad excesiva e irrealista para movilidad urbana (outlier extremo)
events.loc[3, "speed_kmh"] = 180.0

# Velocidad negativa (imposible físicamente: error de medición/cálculo/ETL)
events.loc[10, "speed_kmh"] = -5.0

# -------------------------
# 2) Completitud: metemos campos faltantes (nulls)
# -------------------------

# Falta vehículo (dato incompleto: por ejemplo fallo en el tracking o join a catálogo)
events.loc[6, "vehicle_id"] = None

# Falta precio (pago no registrado, fallo de integración con billing, etc.)
events.loc[11, "price_cents"] = None

# -------------------------
# 3) Consistencia: duplicados por reintentos (mismo event_id repetido)
# -------------------------

# Copiamos una fila existente (la 8) para simular un reintento que inserta el mismo evento dos veces
dupe = events.iloc[[8]].copy()

# Concatenamos el duplicado al final del DataFrame (ahora hay un event_id repetido)
events = pd.concat([events, dupe], ignore_index=True)

# -------------------------
# 4) Errores de zona horaria: alguien guardó hora LOCAL como si fuera UTC (+1)
# -------------------------

# Creamos una columna "ts_mixed" que mezcla datos correctos e incorrectos (anti-patrón)
events["ts_mixed"] = events["ts_utc"].copy()

# Caso erróneo: para un registro, se guarda la hora local (España invierno ~ UTC+1)
# pero se etiqueta como si fuera UTC => queda desplazado +1h respecto a la realidad
events.loc[5, "ts_mixed"] = events.loc[5, "ts_utc"] + timedelta(hours=1)

# Mostramos las primeras 18 filas para inspección rápida
events.head(18)


,event_id,user_id,vehicle_id,ts_utc,speed_kmh,price_cents,event_type,ts_mixed
0,evt_000,102,bike_2,2026-02-01 08:00:00+00:00,28.6,300.0,trip_end,2026-02-01 08:00:00+00:00
1,evt_001,103,scoot_1,2026-02-01 08:05:00+00:00,20.3,300.0,trip_start,2026-02-01 08:05:00+00:00
2,evt_002,104,bike_1,2026-02-01 08:10:00+00:00,28.4,250.0,trip_end,2026-02-01 08:10:00+00:00
3,evt_003,104,bike_1,2026-02-01 08:15:00+00:00,180.0,450.0,trip_start,2026-02-01 08:15:00+00:00
4,evt_004,103,bike_2,2026-02-01 08:20:00+00:00,18.8,300.0,trip_start,2026-02-01 08:20:00+00:00
5,evt_005,102,bike_1,2026-02-01 08:25:00+00:00,14.2,250.0,trip_end,2026-02-01 09:25:00+00:00
6,evt_006,103,None,2026-02-01 08:30:00+00:00,29.0,450.0,trip_end,2026-02-01 08:30:00+00:00
7,evt_007,104,bike_2,2026-02-01 08:35:00+00:00,25.1,200.0,trip_end,2026-02-01 08:35:00+00:00
8,evt_008,103,scoot_1,2026-02-01 08:40:00+00:00,12.2,150.0,trip_start,2026-02-01 08:40:00+00:00
9,evt_009,102,bike_1,2026-02-01 08:45:00+00:00,22.9,300.0,trip_start,2026-02-01 08:45:00+00:00


## 2) Calidad vs fiabilidad (diferencia que cae en exámenes)

- **Calidad:** “¿Este dato es **adecuado para el uso**?”  
  (Un dato puede valer para un informe orientativo y no valer para facturación.)

- **Fiabilidad:** “¿Puedo **confiar en estos datos de forma sostenida** en el tiempo?”  
  (Llegan con regularidad, el significado no cambia sin control, hay trazabilidad, etc.)

En esta práctica mediremos la parte “visible” (calidad) con reglas simples.


## 3) Precisión: detectar valores imposibles o sospechosos

Ejemplos típicos:
- velocidad negativa,
- temperatura fuera de rango,
- estados imposibles (“entregado” antes de “enviado”),
- unidades mal interpretadas (euros vs céntimos).

Vamos a crear reglas sencillas tipo “tests”:
- velocidad entre 0 y 60 km/h (umbral razonable para bici/patinete urbano).


In [ ]:
# =========================
# Regla de calidad (Precisión): velocidad dentro de un rango plausible
# =========================

# Filtramos eventos con velocidades imposibles o poco realistas:
# - speed_kmh < 0  -> físicamente imposible (error de sensor/ETL)
# - speed_kmh > 60 -> muy improbable para movilidad urbana (outlier extremo / dato corrupto)
bad_speed = events[(events["speed_kmh"] < 0) | (events["speed_kmh"] > 60)]

# Mostramos un subconjunto de columnas para auditar rápidamente los casos problemáticos:
# - event_id: identificar el evento
# - ts_utc: cuándo ocurrió (en UTC)
# - speed_kmh: valor sospechoso
# - vehicle_id: vehículo asociado (útil para ver si hay patrón por vehículo)
# - event_type: tipo de evento (por si el error se concentra en start/end)
bad_speed[["event_id", "ts_utc", "speed_kmh", "vehicle_id", "event_type"]]


,event_id,ts_utc,speed_kmh,vehicle_id,event_type
3,evt_003,2026-02-01 08:15:00+00:00,180.0,bike_1,trip_start
10,evt_010,2026-02-01 08:50:00+00:00,-5.0,bike_1,trip_end


In [ ]:
# =========================
# ¿Qué hacemos cuando detectamos velocidades fuera de rango?
# =========================
# Depende del caso de uso y del nivel de confianza para "arreglar" el dato:
# 1) Corregir (data fixing) si conocemos la causa raíz y la transformación es fiable:
#    - Ej.: unidad mal interpretada (mph vs km/h), sensor con escala equivocada, bug conocido en ETL.
#    - Ventaja: recuperas datos para análisis.
#    - Riesgo: si corriges mal, introduces sesgos o errores silenciosos.
#
# 2) Marcar (flag) y tratar aguas abajo:
#    - Añadir una columna booleana (como speed_ok) para filtrar en dashboards/modelos.
#    - Mantener el valor original para auditoría/trazabilidad.
#
# 3) Excluir de métricas / "cuarentena":
#    - Sacar estos eventos de KPIs (p. ej. velocidad media, % uso, etc.).
#    - Enviar registros a una tabla/cola de "quarantine" para revisión o reprocesado.
#
# 4) Alertar / observabilidad:
#    - Si la tasa de errores sube, disparar una alerta (posible regresión del pipeline o sensor).
#
# Aquí aplicamos el enfoque 2: crear un flag de validez y contar cuántos pasan/no pasan.

# Creamos una columna booleana que indica si la velocidad está en el rango [0, 60]
# inclusive="both" incluye 0 y 60 como valores válidos
events["speed_ok"] = events["speed_kmh"].between(0, 60, inclusive="both")

# Contamos cuántos registros son válidos vs inválidos (útil para ratio de calidad)
events["speed_ok"].value_counts()


,count
speed_ok,
True,17
False,2


## 4) Completitud: ¿falta información crítica?

No solo faltan *campos*: también pueden faltar *eventos* o *tramos temporales* completos.

Aquí miramos:
- % de nulos en columnas críticas,
- huecos de tiempo (si esperamos eventos cada X minutos).


In [ ]:
# =========================
# % de nulos por columna (tasa de completitud)
# =========================
# Objetivo: medir completitud. Si una columna tiene muchos nulos, puede:
# - romper joins o reglas de negocio,
# - sesgar métricas (p. ej., precio medio ignorando nulos),
# - indicar fallos de ingesta/ETL o integraciones (tracking, billing, catálogo de vehículos, etc.).
#
# events.isna()        -> DataFrame booleano: True donde hay NaN/None
# .mean()              -> promedio por columna (True=1, False=0) => proporción de nulos
# .sort_values(...)    -> ordena columnas de mayor a menor tasa de nulos
null_rate = events.isna().mean().sort_values(ascending=False)

# Convertimos la proporción a porcentaje y redondeamos a 1 decimal para reporting
(null_rate * 100).round(1)


,0
vehicle_id,5.3
price_cents,5.3
event_id,0.0
ts_utc,0.0
user_id,0.0
speed_kmh,0.0
event_type,0.0
ts_mixed,0.0
speed_ok,0.0


In [ ]:
# =========================
# Detección de huecos temporales usando ts_utc (la columna "ideal")
# =========================
# Objetivo: comprobar continuidad temporal del stream de eventos.
# Si esperamos un evento ~cada 5 minutos, huecos grandes pueden indicar:
# - pérdida de datos (drop en ingesta),
# - retrasos/caídas del dispositivo,
# - problemas de particionado/ventanas,
# - filtrados accidentales en ETL.

# 1) Ordenamos por timestamp y nos quedamos con la serie de tiempos (sin nulos)
# sort_values("ts_utc") garantiza orden cronológico
# ["ts_utc"] selecciona la columna como Serie
# dropna() elimina timestamps faltantes para que diff() tenga sentido
ts = events.sort_values("ts_utc")["ts_utc"].dropna()

# 2) Calculamos diferencias entre timestamps consecutivos
# diff() resta el tiempo anterior al actual: produce una Serie de Timedelta
# dropna() elimina el primer diff (siempre NaT)
gaps = ts.diff().dropna()

# 3) Filtramos huecos "grandes" respecto a lo esperado
# Esperábamos intervalos ~5 min; aquí marcamos como sospechoso > 10 min
# (umbral típico: 2x lo esperado, pero depende del dominio)
big_gaps = gaps[gaps > pd.Timedelta(minutes=10)]

# 4) Devolvemos los huecos grandes (Timedelta) para inspección
# Nota: el índice corresponde al timestamp "actual" (el segundo punto del hueco)
big_gaps

,ts_utc
1,0 days 00:05:00
2,0 days 00:05:00
3,0 days 00:05:00
4,0 days 00:05:00
5,0 days 00:05:00
6,0 days 00:05:00
7,0 days 00:05:00
18,0 days 00:05:00
8,0 days 00:00:00
9,0 days 00:05:00


## 5) Consistencia: duplicados y contradicciones

En distribuido es común “al menos una vez” (reintentos).  
Eso reduce pérdida, pero introduce duplicados si no diseñamos **idempotencia**.

Primero detectamos duplicados por clave:
- si `event_id` debería ser único, no debería repetirse.


In [ ]:
# =========================
# Detección de duplicados por event_id (Consistencia / Unicidad)
# =========================
# Objetivo: encontrar eventos duplicados que suelen aparecer por:
# - reintentos del productor (retry) sin idempotencia,
# - re-procesados en batch,
# - escrituras duplicadas en streaming,
# - merges mal hechos.
#
# events.duplicated(...):
#   - subset=["event_id"] -> define la clave que debería ser única
#   - keep=False -> marca como True TODAS las filas implicadas en el duplicado
#                 (no solo la segunda o última), para ver el grupo completo

# 1) Filtramos las filas cuyo event_id está duplicado y ordenamos por event_id
dupes = events[events.duplicated(subset=["event_id"], keep=False)].sort_values("event_id")

# 2) Mostramos columnas útiles para diagnosticar:
# - event_id: clave duplicada
# - ts_utc: cuándo ocurrió (si son iguales o muy cercanos, puede ser retry)
# - event_type: por si el duplicado afecta solo a start/end
# - user_id, vehicle_id: comprobar si son exactamente la misma entidad o si hay colisiones de IDs
dupes[["event_id", "ts_utc", "event_type", "user_id", "vehicle_id"]]


,event_id,ts_utc,event_type,user_id,vehicle_id
8,evt_008,2026-02-01 08:40:00+00:00,trip_start,103,scoot_1
18,evt_008,2026-02-01 08:40:00+00:00,trip_start,103,scoot_1


### 5.1 Consistencia lógica: precios en céntimos vs euros (error silencioso)

Si alguien interpreta `price_cents=300` como “300 euros”, las métricas explotan.  
Vamos a calcular ingresos de 2 formas:
- correcto: euros = cents/100
- incorrecto: euros = cents (error típico de unidad)


In [ ]:
# =========================
# Impacto de interpretar mal la unidad del precio (céntimos vs euros)
# =========================
# Contexto:
# - price_cents está en CÉNTIMOS (ej.: 150 = 1,50 €)
# - Un error típico es tratar esos valores como si ya estuvieran en euros.
#   Resultado: los ingresos se inflan x100.
#
# Nota: "ignoramos nulls" porque dropna() elimina precios faltantes (NaN/None)
# para no romper la suma y para calcular sobre registros con precio informado.

# Ingreso correcto:
# - Convertimos de céntimos a euros dividiendo por 100
# - Sumamos en euros
revenue_ok = (events["price_cents"].dropna() / 100).sum()

# Ingreso incorrecto:
# - Sumamos los céntimos como si fueran euros (sin convertir)
# - Esto produce un número 100 veces mayor que el real (si la unidad era céntimos)
revenue_wrong = events["price_cents"].dropna().sum()

# Devolvemos ambas cifras para comparar (euros reales vs "euros" mal interpretados)
revenue_ok, revenue_wrong


(np.float64(47.5), np.float64(4750.0))

## 6) Errores de zona horaria: UTC vs local

Esto rompe:
- ventanas por día,
- agregaciones “de ayer”,
- comparaciones entre sistemas.

Tenemos:
- `ts_utc`: lo correcto
- `ts_mixed`: una fila guardada como local pero etiquetada como UTC

Vamos a ver cómo cambia el “día” si agrupamos por fecha.


In [ ]:
# =========================
# Efecto de timestamps mezclados (UTC correcto vs "local guardado como UTC")
# =========================
# Objetivo: ver cómo un error de zona horaria puede cambiar el día asignado a un evento
# y, por tanto, romper agregaciones diarias (KPIs por día, cohortes- grupos que comparten una caracteristica común, facturación diaria, etc.).

# Extraemos la fecha (YYYY-MM-DD) a partir del timestamp en UTC (columna correcta)
# .dt.date devuelve objetos date (pierdes la hora)
events["date_utc"] = events["ts_utc"].dt.date

# Extraemos la fecha a partir del timestamp "mezclado" (algunos registros están desplazados +1h)
events["date_mixed"] = events["ts_mixed"].dt.date

# Comparamos el número de eventos por día usando:
# - date_utc   -> agregación fiable si ts_utc está bien
# - date_mixed -> puede mover eventos de día (por ejemplo cerca de medianoche) y distorsionar conteos
events.groupby("date_utc").size(), events.groupby("date_mixed").size()

# NOTA IMPORTANTE, EN ESTE EJEMPLO SE PODIA FORZAR A QUE UN evento este a las 23:30 UTC y alguien le suma +1h, pasa a 00:30 del día siguiente, y la agregación diaria se rompe.


(date_utc
 2026-02-01    19
 dtype: int64,
 date_mixed
 2026-02-01    19
 dtype: int64)

## 7) Fiabilidad: señales de que “no puedo confiar” aunque hoy parezca bien

Ejemplos típicos de fiabilidad (operativa):
- un día faltan datos (pipeline se cae),
- suben los nulls sin aviso (cambio de esquema),
- cambian cifras históricas por reproceso sin comunicarlo.

Vamos a simular un “cambio de esquema” típico:
- antes venía `speed_kmh`
- ahora viene `speed` (sin coordinación)


In [ ]:
# =========================
# Simulación de "schema drift" (cambio de esquema) en un lote nuevo
# =========================
# Objetivo: emular un caso real de pipelines donde:
# - llega un nuevo lote (batch/partición) con columnas parecidas,
# - pero el productor cambia el esquema sin avisar (p. ej. renombra un campo),
# - lo que rompe transformaciones downstream, validaciones o modelos que esperan "speed_kmh".

# 1) Creamos un nuevo lote tomando 5 filas aleatorias del DataFrame original
# sample(5, random_state=1) -> selección reproducible (siempre las mismas 5 filas)
# copy() -> evitamos "SettingWithCopyWarning" y cambios accidentales sobre events
new_batch = events.sample(5, random_state=1).copy()

# 2) El productor cambia el nombre del campo sin avisar:
# Creamos una columna nueva "speed" copiando el contenido de "speed_kmh"
# (equivalente a un rename, pero hecho en dos pasos para simular un cambio real)
new_batch["speed"] = new_batch["speed_kmh"]

# 3) Eliminamos la columna original "speed_kmh"
# Resultado: el esquema ya no coincide con el esperado por consumidores antiguos
new_batch = new_batch.drop(columns=["speed_kmh"])

# 4) Mostramos las primeras filas para inspección rápida del nuevo esquema
new_batch.head()


,event_id,user_id,vehicle_id,ts_utc,price_cents,event_type,ts_mixed,speed_ok,date_utc,date_mixed,speed
3,evt_003,104,bike_1,2026-02-01 08:15:00+00:00,450.0,trip_start,2026-02-01 08:15:00+00:00,False,2026-02-01,2026-02-01,180.0
15,evt_015,103,bike_2,2026-02-01 09:15:00+00:00,250.0,trip_start,2026-02-01 09:15:00+00:00,True,2026-02-01,2026-02-01,24.1
6,evt_006,103,None,2026-02-01 08:30:00+00:00,450.0,trip_end,2026-02-01 08:30:00+00:00,True,2026-02-01,2026-02-01,29.0
10,evt_010,102,bike_1,2026-02-01 08:50:00+00:00,250.0,trip_end,2026-02-01 08:50:00+00:00,False,2026-02-01,2026-02-01,-5.0
2,evt_002,104,bike_1,2026-02-01 08:10:00+00:00,250.0,trip_end,2026-02-01 08:10:00+00:00,True,2026-02-01,2026-02-01,28.4


In [ ]:
# =========================
# ¿Qué pasa si juntamos ambos lotes sin control de esquema?
# =========================
# Al concatenar (append) DataFrames con columnas distintas, pandas:
# - crea la UNIÓN de columnas (todas las columnas que existan en cualquiera de los dos)
# - rellena con NaN donde una fila no tenga esa columna
#
# En este caso, el lote original tiene "speed_kmh" y el nuevo lote tiene "speed".
# Resultado:
# - en filas de events:   speed_kmh está informado, speed es NaN
# - en filas de new_batch: speed está informado, speed_kmh es NaN
#
# Esto genera nulos "misteriosos" si no sabes que hubo un rename (schema drift),
# y puede romper:
# - métricas (velocidad media),
# - validaciones (parece que faltan datos),
# - modelos/ETL que esperan una sola columna de velocidad.

# 1) Concatenamos ambos lotes sin alinear/normalizar el esquema
# ignore_index=True -> rehace el índice 0..N
# sort=False -> evita reordenar columnas alfabéticamente (mantiene un orden más "natural")
combined = pd.concat([events, new_batch], ignore_index=True, sort=False)

# 2) Calculamos el % de nulos por columna para ver el efecto del schema drift:
# - isna().mean() da la proporción de NaNs (True=1)
# - round(2) para una lectura rápida
combined[["event_id", "speed_kmh", "speed"]].isna().mean().round(2)


,0
event_id,0.00
speed_kmh,0.21
speed,0.79


In [ ]:
# Descomenta para ver lo que paso.
# combined.head(40)

,event_id,user_id,vehicle_id,ts_utc,speed_kmh,price_cents,event_type,ts_mixed,speed_ok,date_utc,date_mixed,speed
0,evt_000,102,bike_2,2026-02-01 08:00:00+00:00,28.6,300.0,trip_end,2026-02-01 08:00:00+00:00,True,2026-02-01,2026-02-01,NaN
1,evt_001,103,scoot_1,2026-02-01 08:05:00+00:00,20.3,300.0,trip_start,2026-02-01 08:05:00+00:00,True,2026-02-01,2026-02-01,NaN
2,evt_002,104,bike_1,2026-02-01 08:10:00+00:00,28.4,250.0,trip_end,2026-02-01 08:10:00+00:00,True,2026-02-01,2026-02-01,NaN
3,evt_003,104,bike_1,2026-02-01 08:15:00+00:00,180.0,450.0,trip_start,2026-02-01 08:15:00+00:00,False,2026-02-01,2026-02-01,NaN
4,evt_004,103,bike_2,2026-02-01 08:20:00+00:00,18.8,300.0,trip_start,2026-02-01 08:20:00+00:00,True,2026-02-01,2026-02-01,NaN
5,evt_005,102,bike_1,2026-02-01 08:25:00+00:00,14.2,250.0,trip_end,2026-02-01 09:25:00+00:00,True,2026-02-01,2026-02-01,NaN
6,evt_006,103,None,2026-02-01 08:30:00+00:00,29.0,450.0,trip_end,2026-02-01 08:30:00+00:00,True,2026-02-01,2026-02-01,NaN
7,evt_007,104,bike_2,2026-02-01 08:35:00+00:00,25.1,200.0,trip_end,2026-02-01 08:35:00+00:00,True,2026-02-01,2026-02-01,NaN
8,evt_008,103,scoot_1,2026-02-01 08:40:00+00:00,12.2,150.0,trip_start,2026-02-01 08:40:00+00:00,True,2026-02-01,2026-02-01,NaN
9,evt_009,102,bike_1,2026-02-01 08:45:00+00:00,22.9,300.0,trip_start,2026-02-01 08:45:00+00:00,True,2026-02-01,2026-02-01,NaN


## 8) Mini-ejercicios

1) **Precisión:** cambia el umbral de velocidad a 40 km/h.  
   - ¿cuántos eventos quedan marcados como malos?

2) **Completitud:** decide qué columnas son “críticas” (mínimo 3) y crea una regla:  
   - “si null% > 1% → alerta”.

3) **Consistencia:** deduplica por `event_id` quedándote con el más reciente (`ts_utc` mayor).  
   - compara el ingreso antes/después.

4) **Zona horaria:** corrige `ts_mixed` suponiendo que **si la diferencia con `ts_utc` es exactamente +1h**, entonces estaba en local.  
   - vuelve a agrupar por día y comprueba si cambia.

5) **Fiabilidad / esquema:** crea una función que “normalice” el esquema (si viene `speed`, crear `speed_kmh`).


In [ ]:
# =========================
# (Opcional) Helper: deduplicar quedándote con el registro "más reciente"
# =========================

def dedup_keep_latest(df, key_cols, ts_col):
    """
    Deduplica un DataFrame por una clave (key_cols) conservando el registro
    con el timestamp más reciente (ts_col) para cada clave.

    Uso típico:
    - Reintentos del productor / doble ingesta: mismo event_id aparece varias veces.
    - Si asumimos que el evento "más nuevo" es el correcto (o el que trae la info final),
      nos quedamos con ese.

    Parámetros:
    - df: DataFrame de entrada
    - key_cols: lista de columnas que definen la unicidad (p. ej. ["event_id"])
    - ts_col: columna timestamp usada para decidir cuál es "el último" (p. ej. "ts_utc")

    Devuelve:
    - DataFrame deduplicado, con índice reseteado.
    """
    # 1) Ordenamos por timestamp ascendente
    # 2) drop_duplicates con keep="last" mantiene la última fila dentro de cada grupo (la de ts más alto)
    # 3) reset_index para dejar un índice limpio tras eliminar filas
    return (df.sort_values(ts_col)
              .drop_duplicates(subset=key_cols, keep="last")
              .reset_index(drop=True))

# Aplicamos deduplicación por event_id usando ts_utc como criterio de "más reciente"
deduped = dedup_keep_latest(events, ["event_id"], "ts_utc")

# =========================
# Impacto en métricas: ingresos antes vs después de deduplicar
# =========================
# before: ingresos calculados con duplicados presentes (puede inflar revenue)
# after: ingresos tras deduplicar (más cercano al "real" si esos duplicados eran reintentos)
before = (events["price_cents"].dropna() / 100).sum()
after = (deduped["price_cents"].dropna() / 100).sum()

# Devolvemos ambas cifras para comparar
before, after


(np.float64(47.5), np.float64(46.0))

## 9) Cierre

- En distribuido, los fallos más caros son **silenciosos**: el job “ok”, el dashboard “bonito”, y aun así el dato está mal.
- La calidad mínima se sostiene midiendo:
  - **precisión** (valores plausibles),
  - **completitud** (no faltan campos/eventos/tiempo),
  - **consistencia** (sin duplicados ni contradicciones).
- La fiabilidad aparece cuando esto ocurre **de forma repetible** y con control de cambios (esquemas, unidades, SLAs).
